# 📄 01 — Extract Text & Detect PII
Point `FILE_PATH` at any PDF, DOCX, PPTX, or image and run all cells.
Everything runs locally — no cloud, no LLM calls.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import yaml
config = yaml.safe_load(open(ROOT / 'config.yaml'))
print('Config loaded. Active profile:', config['active_profile'])

In [ ]:
# Change this to any file you want to process
FILE_PATH = ROOT / 'synthetic' / 'employee_records.docx'

In [ ]:
from extract.router import route
chunks = route(str(FILE_PATH), config)
print(f'Extracted {len(chunks)} chunks from {FILE_PATH.name}')
for c in chunks[:5]:
    status = f'BLOCKED ({c.blocked_reason})' if c.is_blocked() else 'OK'
    print(f'  [{status}] {c.location}: {repr(c.text[:80])}')

In [ ]:
from detect.merge import detect_chunk
import pandas as pd
all_entities = []
for chunk in chunks:
    all_entities.extend(detect_chunk(chunk, config))
print(f'Total PII entities found: {len(all_entities)}')
rows = [
    {
        'Type': e.entity_type,
        'Masked': e.masked_value,
        'Sensitivity': e.sensitivity,
        'Confidence': round(e.confidence, 2),
        'Method': ' | '.join(e.methods),
        'Location': e.location,
    }
    for e in all_entities
]
pd.DataFrame(rows)